# Лабораторная работа №18: Быстрые samplers и distillation компактной diffusion model

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Переиспользовать checkpoint lab 17.
- Сравнить DDPM, DDIM и дополнительный solver.
- Обучить compact student и построить Pareto frontier.

## Теоретическая часть

DDIM использует тот же denoiser, но может идти по сокращённой последовательности timesteps детерминированно. Distillation обучает student воспроизводить несколько teacher steps одним шагом. Сравнение строится при одинаковых initial noise и labels.


DDIM вычисляет x̂0 и делает детерминированный переход между точками ᾱ. DPM-Solver++ использует тот же обученный ε-предсказатель. Progressive distillation аппроксимирует двумя teacher-переходами один student-переход; teacher заморожен.

Детерминированный DDIM-переход: $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\hat\epsilon)/\sqrt{\bar\alpha_t}$, $x_s=\sqrt{\bar\alpha_s}\hat x_0+\sqrt{1-\bar\alpha_s}\hat\epsilon$. При конечном $s=-1$ полагаем $\bar\alpha_{-1}=1$.

### Экспериментальный протокол

Обязательная зависимость — реально обученные веса ddpm.pt, classifier.pt и classifier_validation.json из lab 17. Используйте те же классы, noise и guidance=3 при сравнении. Smoke не является подтверждением качества distillation. Для длинных запусков обеспечьте постоянный каталог artifacts (Drive).

### Материалы

- [DDIM](https://arxiv.org/abs/2010.02502)
- [Progressive Distillation](https://arxiv.org/abs/2202.00512)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.

Версии библиотек, источник данных и лицензия должны быть указаны в отчёте.

## Реализация

`FAST_DEV_RUN=True` проверяет связность пайплайна на малом бюджете и не является итоговым экспериментом. Полная сдача требует `FAST_DEV_RUN=False` и реально полученных метрик.

In [1]:
from pathlib import Path
import os, json, random, time
import numpy as np
import torch
SEED = 42
FAST_DEV_RUN = False  # True только для короткой проверки связности, не для сдачи.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACTS = Path(os.environ.get('CV_ARTIFACTS', 'artifacts'))
RUN_MODE = 'smoke' if FAST_DEV_RUN else 'full'   # smoke и full артефакты физически разделены (протокол лабы)
OUT_DIR = ARTIFACTS / RUN_MODE
ARTIFACTS.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def save_json(name, value):
    destination = OUT_DIR / name   # smoke/full разделены
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, ensure_ascii=False, indent=2))
print({'device':DEVICE,'fast_dev_run':FAST_DEV_RUN,'run_mode':RUN_MODE,'artifacts':str(OUT_DIR)})

{'device': 'cuda', 'fast_dev_run': False, 'run_mode': 'full', 'artifacts': 'artifacts\\full'}


In [2]:
# Сеть: если HTTPS перехватывается прокси/антивирусом (self-signed сертификат),
# переводим проверку сертификатов на системное хранилище ОС (пакет truststore,
# однократно: pip install truststore). Если пакета нет — стандартная проверка.
try:
    import truststore, urllib3
    truststore.inject_into_ssl()
    urllib3.disable_warnings()
    print('SSL: проверка сертификатов через системное хранилище ОС (truststore)')
except ImportError:
    print('SSL: truststore не установлен — стандартная проверка сертификатов')

SSL: проверка сертификатов через системное хранилище ОС (truststore)


In [3]:
# Проверка зависимостей лабы 18: torch, torchvision (MNIST), numpy, matplotlib.
# diffusers НЕ используется - DDIM и DPM-Solver++ реализуются с нуля на том же
# epsilon-предсказателе из лабы 17. Дополнительно включаем детерминизм cudnn:
# сравнение samplers обязано воспроизводиться цифра в цифру.
import importlib.util
_need = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'matplotlib': 'matplotlib'}
missing = [k for k in _need if importlib.util.find_spec(k) is None]
if missing:
    raise ImportError('Установите недостающее и перезапустите ячейку:  pip install ' + ' '.join(_need[k] for k in missing))
import torch, torchvision, numpy, matplotlib
torch.backends.cudnn.deterministic = True   # одинаковые числа при повторном прогоне
torch.backends.cudnn.benchmark = False
_info = {'torch': torch.__version__, 'torchvision': torchvision.__version__,
         'numpy': numpy.__version__, 'matplotlib': matplotlib.__version__,
         'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    _info['gpu'] = f'{_p.name} | cc {_p.major}.{_p.minor}'
else:
    _info['gpu_note'] = 'CUDA недоступна - допустимо, но полный прогон (дистилляция 1500 шагов) на CPU заметно дольше'
print('Версии:', _info)

Версии: {'torch': '2.14.1+cu126', 'torchvision': '0.29.1+cu126', 'numpy': '2.4.6', 'matplotlib': '3.11.2', 'device': 'cuda', 'gpu': 'NVIDIA GeForce GTX 1050 Ti | cc 6.1'}


### Проверка обученного teacher

*Вес: 1 балл.*

**Постановка.** Сравнение sampler допустимо лишь при одинаковой обученной модели, классах и исходном шуме. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите checkpoint lab 17, запретите случайную замену и проверьте совместимость шагов/T.

**Результат.** Проверяемое состояние «Проверка обученного teacher» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите checkpoint lab 17, запретите случайную замену и проверьте совместимость шагов/T» и его результат доступен следующему этапу.

In [4]:
# ЗАДАНИЕ 1 - проверка обученного teacher: загружаем checkpoint лабы 17 (denoiser +
# проверяющий классификатор + его отчёт), запрещаем случайную замену (детерминированный
# поиск файла + sha256 в отчёте) и фиксируем совместимость шагов/T (T берётся ИЗ checkpoint).
# Архитектура ниже совпадает с лабой 17 побитово - strict load не пропустит несовпадение.
import math, hashlib
import torch.nn as nn
import torch.nn.functional as F

BETA_START, BETA_END = 1e-4, 0.02          # тот же линейный schedule, что в лабе 17
TIME_DIM, EMB_DIM = 64, 128
UNCOND_IDX, N_CLASSES_ALL = 10, 11         # 10 цифр + безусловный класс (как в лабе 17)

def diffusion_schedule(t_steps, device=DEVICE):
    betas = torch.linspace(BETA_START, BETA_END, t_steps, device=device)   # [T]
    alphas = 1.0 - betas
    abar = torch.cumprod(alphas, dim=0)                                    # alpha_bar_t = prod alpha_s
    assert bool(torch.all(betas > 0)) and bool(torch.all(betas < 1)), 'beta должна быть в (0,1)'
    assert bool(torch.all(abar[1:] < abar[:-1])) and float(abar[-1]) > 0.0, 'alpha_bar монотонно убывает и > 0'
    return betas, alphas, abar

def q_sample(x0, t, noise):
    # тот же forward-процесс, что в лабе 17: x_t = sqrt(abar_t)*x0 + sqrt(1-abar_t)*eps
    abar_t = ABAR.to(x0.device)[t].view(-1, 1, 1, 1)
    return abar_t.sqrt() * x0 + (1.0 - abar_t).sqrt() * noise

def timestep_embedding(t, dim=TIME_DIM):
    # синусоидальное кодирование timestep (как в Transformer); t: [B] long
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)            # [B, dim]

class ResBlock(nn.Module):
    # residual-блок с инъекцией embedding времени/класса (аддитивная модуляция по каналам)
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(EMB_DIM, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class TinyDenoiser(nn.Module):
    # ТА ЖЕ архитектура, что в лабе 17 (415041 параметров): eps_theta(x_t, t, y)
    def __init__(self, base=32, n_classes=N_CLASSES_ALL):
        super().__init__()
        self.cls_emb = nn.Embedding(n_classes, EMB_DIM)
        self.time_mlp = nn.Sequential(nn.Linear(TIME_DIM, EMB_DIM), nn.SiLU(), nn.Linear(EMB_DIM, EMB_DIM))
        self.stem = nn.Conv2d(1, base, 3, padding=1)
        self.inc = ResBlock(base, base)                                        # 28x28
        self.down1 = nn.Conv2d(base, base, 4, stride=2, padding=1)             # 28 -> 14
        self.mid1 = ResBlock(base, base * 2)                                   # 14x14
        self.down2 = nn.Conv2d(base * 2, base * 2, 4, stride=2, padding=1)     # 14 -> 7
        self.mid2 = ResBlock(base * 2, base * 2)                               # 7x7 - бутылочное горлышко
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                                 nn.Conv2d(base * 2, base * 2, 3, padding=1))  # 7 -> 14
        self.dec2 = ResBlock(base * 4, base)                                   # skip от mid1
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                                 nn.Conv2d(base, base, 3, padding=1))          # 14 -> 28
        self.dec1 = ResBlock(base * 2, base)                                   # skip от inc
        self.out_norm = nn.GroupNorm(8, base)
        self.out_conv = nn.Conv2d(base, 1, 3, padding=1)

    def forward(self, x_t, t, y):
        emb = self.time_mlp(timestep_embedding(t)) + self.cls_emb(y)
        h1 = self.inc(F.silu(self.stem(x_t)), emb)
        h2 = self.mid1(self.down1(h1), emb)
        h3 = self.mid2(self.down2(h2), emb)
        u = self.dec2(torch.cat([self.up2(h3), h2], dim=1), emb)
        u = self.dec1(torch.cat([self.up1(u), h1], dim=1), emb)
        return self.out_conv(F.silu(self.out_norm(u)))

class DigitCNN(nn.Module):
    # ТА ЖЕ архитектура проверяющего классификатора, что в лабе 17 (20490 параметров)
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),    # 28 -> 14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2))   # 14 -> 7
        self.classifier = nn.Linear(32 * 7 * 7, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for blk in iter(lambda: fh.read(chunk), b''):
            h.update(blk)
    return h.hexdigest()

def locate_lab17(names):
    # детерминированный поиск артефакта лабы 17; случайная замена исключена:
    # найденный путь и sha256 фиксируются в отчёте. Порядок поиска:
    # 1) env CV_LAB17_ARTIFACTS;  2) папка ноутбука и вверх до 3 уровней (artifacts/full, artifacts);
    # 3) соседние каталоги (*/*/artifacts и */artifacts - если лаба 18 открыта в отдельной папке);
    # 4) рекурсивный поиск по папке ноутбука.
    roots = []
    if os.environ.get('CV_LAB17_ARTIFACTS'):
        roots.append(Path(os.environ['CV_LAB17_ARTIFACTS']))
    cwd = Path.cwd()
    for base in [cwd, *list(cwd.parents)[:3]]:
        roots += [base, base / 'artifacts', base / 'artifacts' / 'full']
        for _pat in ('*/artifacts', '*/artifacts/full', '*/*/artifacts', '*/*/artifacts/full'):
            roots += [p for p in base.glob(_pat) if p.is_dir()]
    hits = [root / name for root in roots for name in names if (root / name).is_file()]
    if not hits:   # последний шанс - рекурсивный поиск по папке ноутбука
        hits = [p for name in names for p in cwd.rglob(name) if p.is_file()]
    if not hits:
        raise FileNotFoundError(
            'Не найден артефакт лабы 17 (' + ', '.join(names) + '). Проверьте: лаба 17 выполнена в полном '
            'режиме (FAST_DEV_RUN=False) и файлы лежат в artifacts/full. Решения: 1) откройте ноутбук лабы 18 '
            'в той же папке, где запускалась лаба 17;  2) перед этой ячейкой выполните  '
            "import os; os.environ['CV_LAB17_ARTIFACTS'] = r'C:\\cv\\lab17\\artifacts\\full'")
    return max(hits, key=lambda p: p.stat().st_mtime)   # при нескольких совпадениях - самый свежий

def _load(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=True)
    except Exception:
        return torch.load(path, map_location=DEVICE, weights_only=False)  # checkpoint с RNG-состоянием

DDPM_CKPT = locate_lab17(['lab17_ckpt_denoiser.pt', 'ddpm.pt'])
CLF_CKPT = locate_lab17(['lab17_ckpt_classifier.pt', 'classifier.pt'])
CLF_REPORT = locate_lab17(['lab17_classifier_report.json', 'classifier_validation.json'])

_ck = _load(DDPM_CKPT)
assert 'ema' in _ck and 'model' in _ck, 'в checkpoint лабы 17 нет весов/EMA denoiser'
TEACHER_T = int(_ck['T']) if 'T' in _ck else 500    # совместимость шагов: T фиксируется checkpoint'ом
TEACHER_STEP = int(_ck.get('step', _ck.get('steps', -1)))
TEACHER = TinyDenoiser().to(DEVICE)
TEACHER.load_state_dict(_ck['ema'])                 # сэмплирование из EMA-весов (стандарт DDPM)
TEACHER.eval()
for _p in TEACHER.parameters():
    _p.requires_grad_(False)                        # teacher заморожен во всей лабе
N_TEACHER_PARAMS = sum(p.numel() for p in TEACHER.parameters())
_SHA = _sha256(DDPM_CKPT)

_clf_ck = _load(CLF_CKPT)
CLF = DigitCNN().to(DEVICE)
CLF.load_state_dict(_clf_ck['model'])
CLF.eval()
for _p in CLF.parameters():
    _p.requires_grad_(False)
N_CLF_PARAMS = sum(p.numel() for p in CLF.parameters())
CLF_ACC_REPORT = float(json.loads(CLF_REPORT.read_text(encoding='utf-8'))['test_accuracy'])
assert 0.0 < CLF_ACC_REPORT <= 1.0, 'некорректная точность классификатора в отчёте лабы 17'

T_STEPS = TEACHER_T                                  # все samplers лабы работают на T из checkpoint
BETAS, ALPHAS, ABAR = diffusion_schedule(T_STEPS)

def guided_eps(model, x_t, t_scalar, y, w):
    # тот же CFG, что в лабе 17: безусловная ветка всегда, условная добавляется при w>0
    t = torch.full((x_t.size(0),), t_scalar, device=x_t.device, dtype=torch.long)
    eps_u = model(x_t, t, torch.full_like(y, UNCOND_IDX))
    if w == 0:
        return eps_u, 1
    eps_c = model(x_t, t, y)
    return eps_u + w * (eps_c - eps_u), 2

def make_gen(seed):
    # детерминированный генератор на устройстве семплирования (CUDA/CPU)
    return torch.Generator(device=DEVICE).manual_seed(seed) if DEVICE == 'cuda' else torch.Generator().manual_seed(seed)

GUIDANCE_W = 3.0    # протокол лабы 18: сравнение samplers при guidance=3
save_json('lab18_teacher_check.json', {
    'denoiser_ckpt': str(DDPM_CKPT), 'denoiser_sha256': _SHA, 'denoiser_step': TEACHER_STEP,
    'T': T_STEPS, 'T_source': 'checkpoint лабы 17 (случайная замена запрещена)',
    'ema_weights': True, 'teacher_params': N_TEACHER_PARAMS, 'teacher_frozen': True,
    'guidance_w': GUIDANCE_W, 'beta_schedule': f'linear {BETA_START}..{BETA_END} (как в лабе 17)',
    'classifier_ckpt': str(CLF_CKPT), 'classifier_params': N_CLF_PARAMS,
    'classifier_report': str(CLF_REPORT), 'classifier_test_accuracy_report': CLF_ACC_REPORT})
print(f'ЗАДАНИЕ 1 OK: teacher из {DDPM_CKPT.name} (sha256 {_SHA[:12]}..) | step {TEACHER_STEP} | '
      f'T={T_STEPS} взят из checkpoint и зафиксирован | EMA-веса, {N_TEACHER_PARAMS} параметров, заморожен | '
      f'классификатор {N_CLF_PARAMS} параметров, test accuracy по отчёту лабы 17 {CLF_ACC_REPORT:.4f} | w={GUIDANCE_W}')

ЗАДАНИЕ 1 OK: teacher из lab17_ckpt_denoiser.pt (sha256 2eb05e6c7f16..) | step 10000 | T=500 взят из checkpoint и зафиксирован | EMA-веса, 415041 параметров, заморожен | классификатор 20490 параметров, test accuracy по отчёту лабы 17 0.9770 | w=3.0


### Воспроизводимый протокол сравнения

*Вес: 1 балл.*

**Постановка.** Сравнение sampler допустимо лишь при одинаковой обученной модели, классах и исходном шуме. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Создайте общие классы и шум, сохраните seed и качество классификатора.

**Результат.** Проверяемое состояние «Воспроизводимый протокол сравнения» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Создайте общие классы и шум, сохраните seed и качество классификатора» и его результат доступен следующему этапу.

In [5]:
# ЗАДАНИЕ 2 - воспроизводимый протокол сравнения: общие сбалансированные классы, ОБЩИЙ
# исходный шум x_T и общий пошаговый seed для ВСЕХ samplers этого ноутбука; seed и качество
# классификатора фиксируются в отчёте (точность дополнительно перепроверяется своими измерениями).
GEN_PER_CLASS = 1 if FAST_DEV_RUN else 8
GEN_LABELS = torch.arange(10).repeat_interleave(GEN_PER_CLASS)   # сбалансированно: 0,0,...,9,9
GEN_N = GEN_LABELS.size(0)
assert torch.bincount(GEN_LABELS, minlength=10).min().item() == GEN_PER_CLASS, 'классы должны быть сбалансированы'
X_T_SEED, SAMPLING_SEED = SEED + 50, SEED + 51    # те же seed, что в лабе 17 => x_T буквально тот же шум
_noise_g = torch.Generator().manual_seed(X_T_SEED)
X_T_INIT = torch.randn(GEN_N, 1, 28, 28, generator=_noise_g).to(DEVICE)   # одинаковый старт для всех samplers

import torchvision
_mnist_test = torchvision.datasets.MNIST(str(DATA_ROOT / 'mnist'), train=False, download=True)
TEST_N = 200 if FAST_DEV_RUN else 2000
TEST_X = _mnist_test.data[:TEST_N].float().unsqueeze(1) / 255.0   # [N,1,28,28] в [0,1]
TEST_Y = _mnist_test.targets[:TEST_N].long()
assert float(TEST_X.min()) >= 0.0 and float(TEST_X.max()) <= 1.0

@torch.no_grad()
def clf_predict(x01):
    CLF.eval()
    return F.softmax(CLF(x01.to(DEVICE)), dim=1).cpu()

@torch.no_grad()
def clf_eval_test(x_all, y_all):
    # точность классификатора на независимом real test - СВОЁ измерение, а не переписывание отчёта
    probs_all = []
    for i in range(0, x_all.size(0), 512):
        probs_all.append(clf_predict(x_all[i:i + 512]))
    probs = torch.cat(probs_all)
    return float((probs.argmax(1) == y_all).float().mean())

# Санити checkpoint'а: точность меряем на ТЕХ ЖЕ первых 2000 test-изображениях с той же
# нормировкой /255, на которых считался отчёт лабы 17, - и независимо от FAST (это дёшево,
# миллисекунды). Сравнивать с малым FAST-подмножеством (первые 200) некорректно: на нём
# ошибки могут отсутствовать полностью, что даёт фиктивный дрейф 1.0000 vs 0.9770.
_CLF_CHECK_N = 2000
CLF_ACC_MEASURED = clf_eval_test(
    _mnist_test.data[:_CLF_CHECK_N].float().unsqueeze(1) / 255.0,
    _mnist_test.targets[:_CLF_CHECK_N].long())
_drift = abs(CLF_ACC_MEASURED - CLF_ACC_REPORT)
assert _drift < 0.005, (f'точность классификатора разошлась с отчётом лабы 17 '
                        f'({CLF_ACC_MEASURED:.4f} vs {CLF_ACC_REPORT:.4f}) - checkpoint не тот?')
save_json('lab18_protocol.json', {'seed': SEED, 'fast_dev_run': FAST_DEV_RUN, 'guidance_w': GUIDANCE_W,
    'n_images': GEN_N, 'per_class': GEN_PER_CLASS, 'x_T_seed': X_T_SEED, 'sampling_seed': SAMPLING_SEED,
    'labels_bincount': GEN_LABELS.bincount(minlength=10).tolist(), 'teacher_T': T_STEPS,
    'classifier_test_accuracy_report': CLF_ACC_REPORT, 'classifier_test_accuracy_measured': CLF_ACC_MEASURED,
    'classifier_check_n': _CLF_CHECK_N, 'test_n': TEST_N})
print(f'ЗАДАНИЕ 2 OK: {GEN_N} изображений (по {GEN_PER_CLASS} на класс 0..9) | общий x_T (seed {X_T_SEED}) '
      f'и пошаговый seed {SAMPLING_SEED} у всех samplers | guidance w={GUIDANCE_W} | точность классификатора: '
      f'отчёт лабы 17 {CLF_ACC_REPORT:.4f}, перепроверено на тех же {_CLF_CHECK_N} test: {CLF_ACC_MEASURED:.4f}')

ЗАДАНИЕ 2 OK: 80 изображений (по 8 на класс 0..9) | общий x_T (seed 92) и пошаговый seed 93 у всех samplers | guidance w=3.0 | точность классификатора: отчёт лабы 17 0.9770, перепроверено на тех же 2000 test: 0.9770


### Сетка DDIM и граничный переход

*Вес: 1 балл.*

**Постановка.** DDIM сокращает траекторию без переобучения; дополнительный численный solver должен использовать тот же denoiser и CFG. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Реализуйте строго убывающую сетку и DDIM с корректной границей alpha-bar(-1)=1.

**Результат.** Проверяемое состояние «Сетка DDIM и граничный переход» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Реализуйте строго убывающую сетку и DDIM с корректной границей alpha-bar(-1)=1» и его результат доступен следующему этапу.

In [6]:
# ЗАДАНИЕ 3 - сетка DDIM: строго убывающая последовательность timesteps на [0, T-1] и
# детерминированный переход с корректной границей alpha_bar(-1)=1 (финальный шаг отдаёт x0_hat).
def ddim_grid(n, t_steps=T_STEPS):
    # n точек, равномерно по [0, T-1], включая концы; при n=1 - единственный переход T-1 -> (-1)
    n = max(1, min(int(n), int(t_steps)))
    if n == 1:
        return torch.tensor([t_steps - 1], dtype=torch.long)
    grid = torch.linspace(0, t_steps - 1, n).round().long().unique(sorted=True).flip(0)
    assert grid.numel() == n, 'сетка должна содержать ровно n точек'
    assert bool(torch.all(grid[1:] < grid[:-1])), 'сетка обязана быть строго убывающей'
    assert int(grid[0]) == t_steps - 1 and int(grid[-1]) == 0, 'концы сетки: T-1 и 0'
    return grid

def ddim_step(model, x_t, t_scalar, s_scalar, y, w):
    # один детерминированный DDIM-переход t -> s (s=-1 => alpha_bar_s=1: x_s = x0_hat)
    eps, calls = guided_eps(model, x_t, t_scalar, y, w)
    abar_t = ABAR.to(x_t.device)[t_scalar]
    abar_s = torch.tensor(1.0, device=x_t.device) if s_scalar < 0 else ABAR.to(x_t.device)[s_scalar]
    x0_hat = (x_t - (1.0 - abar_t).sqrt() * eps) / abar_t.sqrt()
    x_s = abar_s.sqrt() * x0_hat + (1.0 - abar_s).sqrt() * eps
    return x_s, calls

@torch.no_grad()
def ddim_sample(model, y, w, x_T, grid):
    # полный обратный проход по сетке; NFE = фактические вызовы сети (cond/uncond раздельно)
    model.eval()
    x, nfe = x_T.clone(), 0
    for i in range(grid.numel()):
        t_i = int(grid[i])
        s_i = int(grid[i + 1]) if i + 1 < grid.numel() else -1
        x, calls = ddim_step(model, x, t_i, s_i, y, w)
        nfe += calls
    return x.clamp(0.0, 1.0), nfe

# Проверка формулы на аналитическом примере: подменяем сеть псевдо-сетью с известным выходом
class _EpsStub(nn.Module):
    # детерминированная "сеть": eps = 0.5*tanh(x) + 0.001*y - выход известен аналитически
    def forward(self, x, t, y):
        return 0.5 * torch.tanh(x) + 0.001 * y.float().view(-1, 1, 1, 1)

_chk_grid = ddim_grid(min(3, T_STEPS))
_x = torch.rand(2, 1, 28, 28, generator=torch.Generator().manual_seed(SEED + 1))
_y = torch.tensor([3, UNCOND_IDX])
_x_s, _ = ddim_step(_EpsStub(), _x, int(_chk_grid[0]), int(_chk_grid[1]), _y, GUIDANCE_W)
_eps_u = 0.5 * torch.tanh(_x) + 0.001 * float(UNCOND_IDX)
_eps_c = 0.5 * torch.tanh(_x) + 0.001 * _y.float().view(-1, 1, 1, 1)   # метки y поточечно
_eps_hat = _eps_u + GUIDANCE_W * (_eps_c - _eps_u)             # CFG = eps_u + w*(eps_c - eps_u)
_ab_t, _ab_s = ABAR.cpu()[int(_chk_grid[0])], ABAR.cpu()[int(_chk_grid[1])]
_x0h = (_x - (1.0 - _ab_t).sqrt() * _eps_hat) / _ab_t.sqrt()
_x_ref = _ab_s.sqrt() * _x0h + (1.0 - _ab_s).sqrt() * _eps_hat
assert torch.allclose(_x_s, _x_ref, atol=1e-6), 'формула DDIM-перехода неверна'

# Граница s=-1: alpha_bar(-1)=1 => выход совпадает с чистым x0_hat
_x_end, _ = ddim_step(_EpsStub(), _x, int(_chk_grid[-1]), -1, _y, GUIDANCE_W)
_ab_e = ABAR.cpu()[int(_chk_grid[-1])]
assert torch.allclose(_x_end, (_x - (1.0 - _ab_e).sqrt() * _eps_hat) / _ab_e.sqrt(), atol=1e-6), \
    'граница s=-1 обязана давать чистый x0_hat'

_show = {str(k): ddim_grid(min(k, T_STEPS)).tolist() for k in (20, 10, 5, 1)}
save_json('lab18_ddim_grid.json', {'T': T_STEPS, 'example_grids': _show, 'boundary_abar_minus1': 1.0,
    'formula_check_vs_analytic': 'passed', 'final_step_is_x0_hat': True, 'deterministic': True})
print(f'ЗАДАНИЕ 3 OK: сетка строго убывающая, концы T-1={T_STEPS - 1} и 0 | формула сверена с аналитикой | '
      f'граница s=-1: alpha_bar=1 -> чистый x0_hat | сетка 10 шагов: {_show["10"][:4]}...{_show["10"][-2:]} '
      f'| отчёт: lab18_ddim_grid.json')

ЗАДАНИЕ 3 OK: сетка строго убывающая, концы T-1=499 и 0 | формула сверена с аналитикой | граница s=-1: alpha_bar=1 -> чистый x0_hat | сетка 10 шагов: [499, 444, 388, 333]...[55, 0] | отчёт: lab18_ddim_grid.json


### DPM-Solver++ на той же модели

*Вес: 1 балл.*

**Постановка.** DDIM сокращает траекторию без переобучения; дополнительный численный solver должен использовать тот же denoiser и CFG. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Добавьте solver с тем же epsilon-predictor и beta schedule.

**Результат.** Проверяемое состояние «DPM-Solver++ на той же модели» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Добавьте solver с тем же epsilon-predictor и beta schedule» и его результат доступен следующему этапу.

In [7]:
# ЗАДАНИЕ 4 - DPM-Solver++ (2M, data prediction) на ТОМ ЖЕ epsilon-предсказателе и beta schedule.
# Вывод: x = alpha*x0 + sigma*eps, lambda = log(alpha/sigma), h = lambda_t - lambda_s (h<0 при
# движении к более чистому s). Точный шаг ОДУ с замороженным x0_pred (порядок 1M):
#   x_s = (sigma_s/sigma_t)*x_t + alpha_s*(1 - e^h)*x0_pred        (совпадает с DDIM на общей сетке)
# Порядок 2M экстраполирует x0_pred по двум последним предсказаниям (DPM-Solver++, Lu et al., 2022):
#   D = (1 + 1/(2r))*x0_i - (1/(2r))*x0_{i-1},  r = h_{i-1}/h_i,   x_s = ... + alpha_s*(1 - e^h)*D

def _lam(abar_val):
    # lambda = log(alpha/sigma) = 0.5*log(abar/(1-abar))
    return 0.5 * math.log(abar_val / (1.0 - abar_val))

@torch.no_grad()
def dpm_solver_pp_sample(model, y, w, x_T, grid, order=2):
    # на первом шаге второго предсказания ещё нет - там работает порядок 1M;
    # граница s=-1 (alpha_bar=1): sigma_s=0, alpha_s=1 => точное решение x_s = x0_pred
    model.eval()
    x, nfe = x_T.clone(), 0
    prev_x0, prev_h = None, None
    for i in range(grid.numel()):
        t_i = int(grid[i])
        s_i = int(grid[i + 1]) if i + 1 < grid.numel() else -1
        eps, calls = guided_eps(model, x, t_i, y, w)
        nfe += calls
        abar_t = float(ABAR.to(x.device)[t_i])
        x0 = (x - math.sqrt(1.0 - abar_t) * eps) / math.sqrt(abar_t)
        if s_i < 0:
            x = x0                                   # граница: alpha_bar(-1)=1 -> чистый x0_pred
        else:
            abar_s = float(ABAR.to(x.device)[s_i])
            alpha_t, sigma_t = math.sqrt(abar_t), math.sqrt(1.0 - abar_t)
            alpha_s, sigma_s = math.sqrt(abar_s), math.sqrt(1.0 - abar_s)
            h = _lam(abar_t) - _lam(abar_s)
            if order == 1 or prev_x0 is None:
                d = x0
            else:
                r = prev_h / h
                d = (1.0 + 1.0 / (2.0 * r)) * x0 - (1.0 / (2.0 * r)) * prev_x0
            x = (sigma_s / sigma_t) * x + alpha_s * (1.0 - math.exp(h)) * d
            prev_h = h
        prev_x0 = x0
    return x.clamp(0.0, 1.0), nfe

# Проверка 1: порядок 1M обязан ПОШАГОВО совпасть с DDIM на общей сетке (одно и то же ОДУ)
_g = ddim_grid(min(4, T_STEPS))
_x_in = torch.rand(2, 1, 28, 28, generator=torch.Generator().manual_seed(SEED + 3)).to(DEVICE)
_yv = torch.tensor([1, 7], device=DEVICE)
_img_ddim, _ = ddim_sample(TEACHER, _yv, GUIDANCE_W, _x_in, _g)
_img_1m, _ = dpm_solver_pp_sample(TEACHER, _yv, GUIDANCE_W, _x_in, _g, order=1)
assert torch.allclose(_img_ddim, _img_1m, atol=1e-5), 'DPM-Solver++ 1M обязан совпадать с DDIM на той же сетке'

# Проверка 2: 2M работает и учитывает NFE как DDIM (2 вызова/шаг при w>0)
_img_2m, _nfe_2m = dpm_solver_pp_sample(TEACHER, _yv, GUIDANCE_W, _x_in, _g, order=2)
assert _nfe_2m == 2 * _g.numel(), 'NFE при w>0 обязан быть 2 вызова на шаг'
_diff = float((_img_2m - _img_ddim).abs().max())
save_json('lab18_dpm_solver.json', {'implementation': 'DPM-Solver++ 2M (data prediction, Lu et al. 2022)',
    'epsilon_predictor': 'тот же TinyDenoiser (EMA) из лабы 17', 'beta_schedule': 'linear 1e-4..0.02 (как в лабе 17)',
    'check_1m_equals_ddim': True, 'max_abs_diff_2m_vs_ddim': round(_diff, 6), 'check_grid_steps': int(_g.numel()),
    'nfe_w3': _nfe_2m})
print(f'ЗАДАНИЕ 4 OK: DPM-Solver++(2M) на том же denoiser и beta schedule | 1M == DDIM на общей сетке (проверено) | '
      f'2M vs DDIM на {_g.numel()} шагах: max |diff| {_diff:.4f} | NFE {_nfe_2m} | отчёт: lab18_dpm_solver.json')

ЗАДАНИЕ 4 OK: DPM-Solver++(2M) на том же denoiser и beta schedule | 1M == DDIM на общей сетке (проверено) | 2M vs DDIM на 4 шагах: max |diff| 1.0000 | NFE 8 | отчёт: lab18_dpm_solver.json


### Инварианты детерминизма sampler

*Вес: 1 балл.*

**Постановка.** DDIM сокращает траекторию без переобучения; дополнительный численный solver должен использовать тот же denoiser и CFG. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Проверьте конец сетки и совпадение двух запусков DDIM при одинаковом шуме.

**Результат.** Проверяемое состояние «Инварианты детерминизма sampler» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Проверьте конец сетки и совпадение двух запусков DDIM при одинаковом шуме» и его результат доступен следующему этапу.

In [8]:
# ЗАДАНИЕ 5 - инварианты детерминизма: конец сетки (T-1 ... 0, строго убывающая) и ПОЛНОЕ
# совпадение двух запусков DDIM при одинаковом шуме; дополнительно фиксируем, что
# детерминированный sampler вообще не потребляет RNG (состояние генератора не меняется).
_grid = ddim_grid(min(20, T_STEPS))
assert int(_grid[0]) == T_STEPS - 1 and int(_grid[-1]) == 0, 'конец сетки: T-1 и 0'
_rng_before = torch.get_rng_state()
_cuda_before = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
_t0 = time.time()
_img_a, _nfe_a = ddim_sample(TEACHER, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT, _grid)
_time_a = round(time.time() - _t0, 1)
_t0 = time.time()
_img_b, _nfe_b = ddim_sample(TEACHER, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT, _grid)
_time_b = round(time.time() - _t0, 1)
_rng_after = torch.get_rng_state()
_cuda_after = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
assert torch.equal(_img_a, _img_b), 'повторный запуск DDIM при том же шуме обязан дать те же изображения'
assert _nfe_a == _nfe_b == 2 * _grid.numel(), 'NFE DDIM при w=3 обязан быть 2 вызова на шаг'
assert torch.equal(_rng_before, _rng_after), 'DDIM (eta=0) не должен потреблять CPU-RNG'
_cuda_same = True if _cuda_before is None else all(torch.equal(a, b) for a, b in zip(_cuda_before, _cuda_after))
_dev_rng = 'CPU' if _cuda_before is None else 'CPU + CUDA'
save_json('lab18_determinism.json', {'grid_first': int(_grid[0]), 'grid_last': int(_grid[-1]),
    'strictly_decreasing': True, 'grid_steps': int(_grid.numel()), 'repeat_equal': True,
    'rng_state_unchanged': bool(torch.equal(_rng_before, _rng_after)), 'cuda_rng_unchanged': _cuda_same,
    'nfe': _nfe_a, 'time_s': [_time_a, _time_b]})
print(f'ЗАДАНИЕ 5 OK: два запуска DDIM (сетка {_grid.numel()} шагов, NFE {_nfe_a}) дали побитово одинаковые '
      f'изображения | RNG не потрачен ({_dev_rng}) | '
      f'время {_time_a} c / {_time_b} c | отчёт: lab18_determinism.json')

ЗАДАНИЕ 5 OK: два запуска DDIM (сетка 20 шагов, NFE 40) дали побитово одинаковые изображения | RNG не потрачен (CPU + CUDA) | время 0.8 c / 0.7 c | отчёт: lab18_determinism.json


### Измеримые бюджеты sampling

*Вес: 1 балл.*

**Постановка.** Время и качество измеряются на одном batch; количество NFE учитывает conditional и unconditional обращения отдельно. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Измерьте class agreement и внутриклассовое разнообразие PNG при равном начальном шуме, учитывайте NFE.

**Результат.** Проверяемое состояние «Измеримые бюджеты sampling» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Измерьте class agreement и внутриклассовое разнообразие PNG при равном начальном шуме, учитывайте NFE» и его результат доступен следующему этапу.

In [9]:
# ЗАДАНИЕ 6 - измеримые бюджеты sampling: на ОДНОМ batch (общие x_T/labels, seed 93) сравниваем
# DDPM (полная марковская цепь), DDIM и DPM-Solver++(2M) на сетках разной длины; для каждого
# прогона измеряем class agreement, внутриклассовое разнообразие, уверенность, время и NFE
# (conditional и unconditional вызовы учитываются раздельно: при w=3 это 2 вызова/шаг).
import matplotlib.pyplot as plt

def ddpm_sample(model, y, w, x_T, steps, gen_seed):
    # полная стохастическая цепь DDPM с фиксированным seed пошагового шума z (как в лабе 17)
    model.eval()
    x, nfe = x_T.clone(), 0
    g = make_gen(gen_seed)
    for t in range(T_STEPS - 1, -1, -1):
        eps, calls = guided_eps(model, x, t, y, w)
        nfe += calls
        mean = (x - (1.0 - ALPHAS[t]) / (1.0 - ABAR[t]).sqrt() * eps) / ALPHAS[t].sqrt()
        if t > 0:
            z = torch.randn(x.shape, generator=g, device=x.device)
            x = mean + BETAS[t].sqrt() * z
    return x.clamp(0.0, 1.0), nfe

def intra_class_diversity(imgs, labels):
    # средняя попарная L2 ВНУТРИ каждого класса (нужно >=2 сэмпла на класс)
    if int(labels.bincount(minlength=10).min()) < 2:
        return None
    flat = imgs.reshape(imgs.size(0), -1)
    vals = []
    for c in range(10):
        idx = (labels == c).nonzero(as_tuple=True)[0]
        if idx.numel() < 2:
            continue
        d = torch.cdist(flat[idx], flat[idx])
        iu = torch.triu_indices(idx.numel(), idx.numel(), offset=1)
        vals.append(float(d[iu[0], iu[1]].mean()))
    return sum(vals) / len(vals)

def agreement_conf(imgs):
    probs = clf_predict(imgs)
    return float((probs.argmax(1) == GEN_LABELS).float().mean()), float(probs.max(1).values.mean())

def save_grid(imgs, path, per_class):
    # строки = классы 0..9, столбцы = сэмплы класса
    f, ax = plt.subplots(10, per_class, figsize=(max(1.2 * per_class, 2.5), 12.5),
                         squeeze=False, constrained_layout=True)
    for c in range(10):
        for j in range(per_class):
            ax[c, j].imshow(imgs[c * per_class + j, 0], cmap='gray', vmin=0, vmax=1)
            ax[c, j].axis('off')
        ax[c, 0].text(-0.08, 0.5, str(c), transform=ax[c, 0].transAxes, ha='right', va='center', fontsize=8)
    f.savefig(path, dpi=110)
    plt.close(f)

SAMPLER_BUDGETS = [2, 1] if FAST_DEV_RUN else [50, 20, 10, 5]
DDPM_REF_STEPS = min(2, T_STEPS) if FAST_DEV_RUN else T_STEPS
BUDGET_RUNS = []

def run_budget(sampler, steps):
    _t0 = time.time()
    if sampler == 'ddpm':
        imgs, nfe = ddpm_sample(TEACHER, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT, steps, SAMPLING_SEED)
    elif sampler == 'ddim':
        imgs, nfe = ddim_sample(TEACHER, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT, ddim_grid(steps))
    else:
        imgs, nfe = dpm_solver_pp_sample(TEACHER, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT, ddim_grid(steps), order=2)
    imgs = imgs.cpu()
    _agr, _conf = agreement_conf(imgs)
    _div = intra_class_diversity(imgs, GEN_LABELS)
    rec = {'sampler': sampler, 'steps': steps, 'agreement': round(_agr, 4), 'confidence': round(_conf, 4),
           'diversity_intra_l2': None if _div is None else round(_div, 3),
           'time_s': round(time.time() - _t0, 1), 'nfe': nfe}
    save_grid(imgs, OUT_DIR / f'lab18_budget_{sampler}_s{steps}.png', GEN_PER_CLASS)
    BUDGET_RUNS.append(rec)
    _d = 'n/a (1 сэмпл/класс)' if _div is None else f'{_div:.3f}'
    print(f'  {sampler:5s} шагов={steps:3d}: agreement {_agr:.3f} | уверенность {_conf:.3f} | '
          f'внутриклассовое L2 {_d} | NFE {nfe} | {rec["time_s"]} c')

run_budget('ddpm', DDPM_REF_STEPS)
for _st in SAMPLER_BUDGETS:
    run_budget('ddim', _st)
    run_budget('dspp', _st)
save_json('lab18_budgets.json', {'runs': BUDGET_RUNS, 'guidance_w': GUIDANCE_W, 'x_T_seed': X_T_SEED,
    'sampling_seed': SAMPLING_SEED, 'ddpm_steps': DDPM_REF_STEPS, 'images': GEN_N,
    'note': 'NFE учитывает conditional и unconditional вызовы отдельно (w=3 -> 2 вызова/шаг); '
            'у всех прогонов общий x_T и общий seed пошагового шума'})
print(f'ЗАДАНИЕ 6 OK: {len(BUDGET_RUNS)} прогонов на общем шуме и общих классах '
      f'(x_T seed {X_T_SEED}, z seed {SAMPLING_SEED}) | PNG: lab18_budget_*.png | отчёт: lab18_budgets.json')

  ddpm  шагов=500: agreement 0.988 | уверенность 0.991 | внутриклассовое L2 8.002 | NFE 1000 | 18.0 c
  ddim  шагов= 50: agreement 1.000 | уверенность 1.000 | внутриклассовое L2 8.123 | NFE 100 | 1.8 c
  dspp  шагов= 50: agreement 1.000 | уверенность 1.000 | внутриклассовое L2 8.307 | NFE 100 | 1.8 c
  ddim  шагов= 20: agreement 1.000 | уверенность 1.000 | внутриклассовое L2 7.899 | NFE 40 | 0.7 c
  dspp  шагов= 20: agreement 1.000 | уверенность 0.999 | внутриклассовое L2 8.417 | NFE 40 | 0.7 c
  ddim  шагов= 10: agreement 1.000 | уверенность 1.000 | внутриклассовое L2 7.537 | NFE 20 | 0.4 c
  dspp  шагов= 10: agreement 1.000 | уверенность 0.999 | внутриклассовое L2 8.511 | NFE 20 | 0.4 c
  ddim  шагов=  5: agreement 1.000 | уверенность 1.000 | внутриклассовое L2 6.735 | NFE 10 | 0.2 c
  dspp  шагов=  5: agreement 0.962 | уверенность 0.958 | внутриклассовое L2 8.663 | NFE 10 | 0.2 c
ЗАДАНИЕ 6 OK: 9 прогонов на общем шуме и общих классах (x_T seed 92, z seed 93) | PNG: lab18_budget_*.pn

### Pareto скорости и качества

*Вес: 1 балл.*

**Постановка.** Время и качество измеряются на одном batch; количество NFE учитывает conditional и unconditional обращения отдельно. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Определите недоминируемые точки только из измеренных времени и точности.

**Результат.** Проверяемое состояние «Pareto скорости и качества» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Определите недоминируемые точки только из измеренных времени и точности» и его результат доступен следующему этапу.

In [10]:
# ЗАДАНИЕ 7 - Pareto скорости и качества: недоминируемые точки определяются ТОЛЬКО по
# фактически измеренным времени (меньше - лучше) и class agreement (больше - лучше) из Задания 6.
_points = [{'name': f"{r['sampler']}-s{r['steps']}", 'time_s': r['time_s'], 'agreement': r['agreement']}
           for r in BUDGET_RUNS]

def pareto_front(points):
    # A доминирует B при time_A<=time_B, agreement_A>=agreement_B и хотя бы одном строгом неравенстве
    front = []
    for a in points:
        dominated = any((b['time_s'] <= a['time_s'] and b['agreement'] >= a['agreement']) and
                        (b['time_s'] < a['time_s'] or b['agreement'] > a['agreement']) for b in points)
        if not dominated:
            front.append(a)
    return sorted(front, key=lambda p: (p['time_s'], p['agreement']))

_front = pareto_front(_points)
_front_names = {p['name'] for p in _front}
save_json('lab18_pareto.json', {'points': _points, 'pareto_front': _front,
    'rule': 'A доминирует B при time_A<=time_B, agreement_A>=agreement_B и хотя бы одном строгом неравенстве'})

import matplotlib.pyplot as plt
_f, _ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for p in _points:
    _is_front = p['name'] in _front_names
    _ax.scatter(p['time_s'], p['agreement'], s=80 if _is_front else 34,
                marker='o' if _is_front else 'x', color='#1f77b4' if _is_front else 'gray',
                zorder=3 if _is_front else 2)
    _ax.annotate(p['name'], (p['time_s'], p['agreement']), textcoords='offset points',
                 xytext=(6, 4), fontsize=7)
_fx = [p['time_s'] for p in _front]
_fy = [p['agreement'] for p in _front]
_ax.plot(_fx, _fy, '--', color='#1f77b4', lw=1.2, zorder=1, label='Pareto front')
_ax.set_xscale('log')
_ax.set_xlabel('время генерации 80 изображений, c (log scale)')
_ax.set_ylabel('class agreement')
_ax.set_ylim(-0.02, 1.02)
_ax.set_title(f'Pareto samplers: DDPM / DDIM / DPM-Solver++(2M), w={GUIDANCE_W}, общий x_T')
_ax.legend(fontsize=8)
_f.savefig(OUT_DIR / 'lab18_pareto.png', dpi=120)
plt.close(_f)
print('ЗАДАНИЕ 7 OK: недоминируемые точки (только измеренные время/agreement): ' +
      ', '.join(p['name'] for p in _front) + ' | PNG: lab18_pareto.png | отчёт: lab18_pareto.json')

ЗАДАНИЕ 7 OK: недоминируемые точки (только измеренные время/agreement): ddim-s5 | PNG: lab18_pareto.png | отчёт: lab18_pareto.json


### Подготовка отдельного student

*Вес: 1 балл.*

**Постановка.** Student обучается на последовательных переходах teacher, а не на произвольной регрессии к фиксированному изображению. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите train MNIST и инициализируйте отдельную модель весами teacher.

**Результат.** Проверяемое состояние «Подготовка отдельного student» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите train MNIST и инициализируйте отдельную модель весами teacher» и его результат доступен следующему этапу.

In [11]:
# ЗАДАНИЕ 8 - подготовка отдельного student: train MNIST (те же фиксированные префиксы, что
# в лабе 17) и ОТДЕЛЬНАЯ модель, инициализированная весами teacher; teacher остаётся замороженным.
import copy
import torchvision
_mnist_train = torchvision.datasets.MNIST(str(DATA_ROOT / 'mnist'), train=True, download=True)
TRAIN_N = 200 if FAST_DEV_RUN else 20000
TRAIN_X = _mnist_train.data[:TRAIN_N].float().unsqueeze(1) / 255.0    # [N,1,28,28] в [0,1]
TRAIN_Y = _mnist_train.targets[:TRAIN_N].long()
assert float(TRAIN_X.min()) >= 0.0 and float(TRAIN_X.max()) <= 1.0

STUDENT = TinyDenoiser().to(DEVICE)
STUDENT.load_state_dict(TEACHER.state_dict())       # инициализация весами teacher (EMA)
assert STUDENT is not TEACHER, 'student обязан быть отдельным объектом, а не ссылкой на teacher'
_max_diff = max(float((a - b).detach().abs().max()) for a, b in zip(STUDENT.parameters(), TEACHER.parameters()))
assert _max_diff == 0.0, 'при инициализации веса student обязаны совпадать с весами teacher'
STUDENT.train()
assert all(not p.requires_grad for p in TEACHER.parameters()), 'teacher должен остаться замороженным'
N_STUDENT_PARAMS = sum(p.numel() for p in STUDENT.parameters())
save_json('lab18_student_init.json', {'init': 'веса teacher (EMA) из лабы 17', 'student_params': N_STUDENT_PARAMS,
    'max_weight_diff_init': _max_diff, 'teacher_frozen': True, 'train_n': TRAIN_N,
    'train_classes': TRAIN_Y.bincount(minlength=10).tolist()})
print(f'ЗАДАНИЕ 8 OK: student = TinyDenoiser {N_STUDENT_PARAMS} параметров, отдельный объект, '
      f'инициализирован весами teacher (max |diff| {_max_diff}) | teacher заморожен | '
      f'train MNIST {tuple(TRAIN_X.shape)} (фиксированный префикс, как в лабе 17)')

ЗАДАНИЕ 8 OK: student = TinyDenoiser 415041 параметров, отдельный объект, инициализирован весами teacher (max |diff| 0.0) | teacher заморожен | train MNIST (20000, 1, 28, 28) (фиксированный префикс, как в лабе 17)


### Двухшаговые цели для distillation

*Вес: 1 балл.*

**Постановка.** Student обучается на последовательных переходах teacher, а не на произвольной регрессии к фиксированному изображению. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Оптимизируйте student по двум переходам замороженного teacher с checkpoint.

**Результат.** Проверяемое состояние «Двухшаговые цели для distillation» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Оптимизируйте student по двум переходам замороженного teacher с checkpoint» и его результат доступен следующему этапу.

In [12]:
# ЗАДАНИЕ 9 - двухшаговые цели progressive distillation: student ОДНИМ детерминированным
# DDIM-переходом аппроксимирует ДВА последовательных DDIM-перехода замороженного teacher
# (t -> m -> s при guidance w=3), а не регрессию к фиксированному изображению.
# Сетка целей DG (20 точек в полном режиме) => пар переходов N_PAIRS; student потом семплирует
# по чётным точкам DG (каждый его переход - ровно то, чему он учился). Чекпоинт + resume.
class EMA:
    # экспоненциальное скользящее среднее весов (как в лабе 17); сэмплирование - из EMA-копии
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[k].copy_(v)

DISTILL_STEPS = 4 if FAST_DEV_RUN else 20           # точек в сетке teacher-целей
DISTILL_BATCH = 16 if FAST_DEV_RUN else 128
DISTILL_LR = 2e-4
DISTILL_STEPS_TRAIN = 2 if FAST_DEV_RUN else 1500   # шагов оптимизации student

DG = ddim_grid(DISTILL_STEPS)                       # строго убывающая сетка [T-1 ... 0]
N_PAIRS = DG.numel() // 2
assert N_PAIRS >= 1, 'сетка дистилляции обязана содержать хотя бы одну пару переходов'

def distill_batches(seed, n, steps, batch):
    # детерминированный поток батчей (как в лабе 17): одинаковый при любом перезапуске
    g = torch.Generator().manual_seed(seed)
    return [torch.randint(0, n, (batch,), generator=g) for _ in range(steps)]

S_OPT = torch.optim.Adam(STUDENT.parameters(), lr=DISTILL_LR)
_s_ckpt = OUT_DIR / 'lab18_ckpt_student.pt'   # smoke/full разделены; раньше путь был общий -
                                              # из-за этого resume подхватывал чекпоинт чужого режима
if (ARTIFACTS / 'lab18_ckpt_student.pt').is_file():
    print('ВНИМАНИЕ: старый student checkpoint artifacts/lab18_ckpt_student.pt (общий путь) игнорирую: '
          'smoke/full не смешиваются; актуальный путь:', str(_s_ckpt))
_s_batches = distill_batches(SEED + 60, TRAIN_N, DISTILL_STEPS_TRAIN, DISTILL_BATCH)
_s_ema = EMA(STUDENT)
_s_hist, _start = [], 0
if _s_ckpt.exists():
    _ck = _load(_s_ckpt)
    STUDENT.load_state_dict(_ck['model']); S_OPT.load_state_dict(_ck['optimizer'])
    _s_ema.shadow = {k: v.to(DEVICE) for k, v in _ck['ema'].items()}
    _start, _s_hist = _ck['step'], _ck['loss_history']
    assert _ck.get('grid') == DG.tolist(), 'сетка дистилляции изменилась - student checkpoint несовместим'
    torch.set_rng_state(_ck['rng']['cpu'].cpu())
    if _ck['rng'].get('cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in _ck['rng']['cuda']])
    print('student: resume с шага', _start)

STUDENT.train()
_t0 = time.time()
_log_every = max(1, DISTILL_STEPS_TRAIN // 10)
for _step in range(_start, DISTILL_STEPS_TRAIN):
    _x0 = TRAIN_X[_s_batches[_step]].to(DEVICE)
    _y = TRAIN_Y[_s_batches[_step]].clone().to(DEVICE)
    _k = _step % N_PAIRS                            # детерминированный перебор пар переходов
    _t_i = int(DG[2 * _k])
    _s_mid = int(DG[2 * _k + 1])
    _s_end = int(DG[2 * _k + 2]) if 2 * _k + 2 < DG.numel() else -1
    _g = torch.Generator().manual_seed(SEED + 70 + _step)   # свой seed на шаг: точный resume
    _eps = torch.randn(_x0.shape, generator=_g).to(DEVICE)
    _x_t = q_sample(_x0, torch.full((_x0.size(0),), _t_i, dtype=torch.long, device=DEVICE), _eps)
    with torch.no_grad():                           # ЦЕЛЬ: два DDIM-перехода замороженного teacher
        _x_m, _ = ddim_step(TEACHER, _x_t, _t_i, _s_mid, _y, GUIDANCE_W)
        _x_target, _ = ddim_step(TEACHER, _x_m, _s_mid, _s_end, _y, GUIDANCE_W)
    _x_pred, _ = ddim_step(STUDENT, _x_t, _t_i, _s_end, _y, GUIDANCE_W)   # один переход student
    _loss = F.mse_loss(_x_pred, _x_target)
    S_OPT.zero_grad(); _loss.backward(); S_OPT.step()
    _s_ema.update(STUDENT)
    _s_hist.append(float(_loss.item()))
    if (_step + 1) % _log_every == 0 or _step + 1 == DISTILL_STEPS_TRAIN:
        _recent = _s_hist[-50:]
        print(f'  distill шаг {_step + 1}/{DISTILL_STEPS_TRAIN} | MSE avg50 {sum(_recent) / len(_recent):.4f} | {time.time() - _t0:.0f} c')
DISTILL_TIME = time.time() - _t0
_rng = {'cpu': torch.get_rng_state()}
if torch.cuda.is_available():
    _rng['cuda'] = torch.cuda.get_rng_state_all()
torch.save({'model': STUDENT.state_dict(), 'optimizer': S_OPT.state_dict(), 'ema': _s_ema.shadow,
            'step': DISTILL_STEPS_TRAIN, 'loss_history': _s_hist, 'grid': DG.tolist(),
            'steps': DISTILL_STEPS_TRAIN, 'lr': DISTILL_LR, 'guidance_w': GUIDANCE_W, 'rng': _rng}, _s_ckpt)
save_json('lab18_distillation.json', {'steps': DISTILL_STEPS_TRAIN, 'batch': DISTILL_BATCH, 'lr': DISTILL_LR,
    'teacher_grid': DG.tolist(), 'n_pairs': N_PAIRS, 'guidance_w': GUIDANCE_W, 'T': T_STEPS,
    'loss_first': _s_hist[0], 'loss_last': _s_hist[-1], 'history_every_50': _s_hist[::50],
    'train_time_s': round(DISTILL_TIME, 1), 'checkpoint': _s_ckpt.name})
print(f'ЗАДАНИЕ 9 OK: student дистиллирован за {DISTILL_STEPS_TRAIN} шагов: MSE {_s_hist[0]:.4f} -> {_s_hist[-1]:.4f} '
      f'(цель: 2 DDIM-перехода teacher w={GUIDANCE_W} -> 1 переход student; сетка {DG.numel()} точек, {N_PAIRS} пар) '
      f'| {DISTILL_TIME:.1f} c | чекпоинт: {_s_ckpt.name}')

ВНИМАНИЕ: старый student checkpoint artifacts/lab18_ckpt_student.pt (общий путь) игнорирую: smoke/full не смешиваются; актуальный путь: artifacts\full\lab18_ckpt_student.pt
  distill шаг 150/1500 | MSE avg50 0.0004 | 49 c
  distill шаг 300/1500 | MSE avg50 0.0006 | 98 c
  distill шаг 450/1500 | MSE avg50 0.0003 | 147 c
  distill шаг 600/1500 | MSE avg50 0.0003 | 197 c
  distill шаг 750/1500 | MSE avg50 0.0004 | 247 c
  distill шаг 900/1500 | MSE avg50 0.0003 | 296 c
  distill шаг 1050/1500 | MSE avg50 0.0003 | 345 c
  distill шаг 1200/1500 | MSE avg50 0.0009 | 394 c
  distill шаг 1350/1500 | MSE avg50 0.0004 | 443 c
  distill шаг 1500/1500 | MSE avg50 0.0005 | 492 c
ЗАДАНИЕ 9 OK: student дистиллирован за 1500 шагов: MSE 0.0000 -> 0.0001 (цель: 2 DDIM-перехода teacher w=3.0 -> 1 переход student; сетка 20 точек, 10 пар) | 492.1 c | чекпоинт: lab18_ckpt_student.pt


### Сравнение при равных и разных NFE

*Вес: 1 балл.*

**Постановка.** Student обучается на последовательных переходах teacher, а не на произвольной регрессии к фиксированному изображению. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сравните teacher 20/10 и student 10 на общих noise/labels; сохраните изображения и время.

**Результат.** Проверяемое состояние «Сравнение при равных и разных NFE» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сравните teacher 20/10 и student 10 на общих noise/labels; сохраните изображения и время» и его результат доступен следующему этапу.

In [13]:
# ЗАДАНИЕ 10 - сравнение при равных и разных NFE: teacher DDIM 20/10 шагов против student 10
# шагов на ОБЩИХ noise/labels (x_T seed 92, w=3). Student семплирует по чётным точкам сетки
# дистилляции - его переходы в точности совпадают с теми, чему он учился. Сохраняем изображения,
# время и близость к teacher-20; отдельно - пять слабейших примеров для раздела "Анализ результатов".
SAMPLE_STUDENT = copy.deepcopy(STUDENT)
SAMPLE_STUDENT.load_state_dict(_s_ema.shadow)       # сэмплируем из EMA-весов student
SAMPLE_STUDENT.eval().to(DEVICE)

TEACH_GRID_20 = ddim_grid(4 if FAST_DEV_RUN else 20)
TEACH_GRID_10 = ddim_grid(2 if FAST_DEV_RUN else 10)
STUDENT_GRID = DG[0::2]                             # g0 -> g2 -> ... -> (-1): ровно обученные переходы
assert STUDENT_GRID.numel() == DG.numel() // 2

def _run(model, grid):
    _t0 = time.time()
    imgs, nfe = ddim_sample(model, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT, grid)
    return imgs.cpu(), {'steps': int(grid.numel()), 'nfe': nfe, 'time_s': round(time.time() - _t0, 1)}

_t20, _r20 = _run(TEACHER, TEACH_GRID_20)
_t10, _r10 = _run(TEACHER, TEACH_GRID_10)
_stu, _rs = _run(SAMPLE_STUDENT, STUDENT_GRID)

def _metrics(imgs):
    _agr, _conf = agreement_conf(imgs)
    _div = intra_class_diversity(imgs, GEN_LABELS)
    return {'agreement': round(_agr, 4), 'confidence': round(_conf, 4),
            'diversity_intra_l2': None if _div is None else round(_div, 3)}

_m20, _m10, _ms = _metrics(_t20), _metrics(_t10), _metrics(_stu)
_flat = lambda a: a.reshape(a.size(0), -1)
_l2_stu = float(torch.cdist(_flat(_stu), _flat(_t20)).mean())   # близость student к teacher-20
_l2_t10 = float(torch.cdist(_flat(_t10), _flat(_t20)).mean())   # близость teacher-10 к teacher-20
for _imgs, _nm in [(_t20, 'teacher20'), (_t10, 'teacher10'), (_stu, 'student10')]:
    save_grid(_imgs, OUT_DIR / f'lab18_{_nm}.png', GEN_PER_CLASS)

# пять слабейших примеров student-10 (минимальная уверенность классификатора) - для анализа ошибок
_probs = clf_predict(_stu)
_conf, _pred = _probs.max(1)
_worst_idx = _conf.argsort()[:5].tolist()
_worst = [{'index': _i, 'target': int(GEN_LABELS[_i]), 'predicted': int(_pred[_i]),
           'confidence': round(float(_conf[_i]), 4)} for _i in _worst_idx]
_f, _ax = plt.subplots(1, 5, figsize=(9.5, 2.4), constrained_layout=True)
for _axi, _i in zip(_ax, _worst_idx):
    _axi.imshow(_stu[_i, 0], cmap='gray', vmin=0, vmax=1)
    _axi.set_title(f'y={int(GEN_LABELS[_i])} pred={int(_pred[_i])} conf={float(_conf[_i]):.2f}', fontsize=7)
    _axi.axis('off')
_f.savefig(OUT_DIR / 'lab18_worst_generated.png', dpi=120)
plt.close(_f)

save_json('lab18_student_vs_teacher.json', {
    'runs': [{'name': f'teacher-{_r20["steps"]}', **_r20, **_m20},
             {'name': f'teacher-{_r10["steps"]}', **_r10, **_m10},
             {'name': f'student-{_rs["steps"]}', **_rs, **_ms}],
    'l2_to_teacher20_mean': {'student10': round(_l2_stu, 3), 'teacher10': round(_l2_t10, 3)},
    'same_noise_labels': True, 'x_T_seed': X_T_SEED, 'guidance_w': GUIDANCE_W, 'worst5': _worst})
print(f'ЗАДАНИЕ 10 OK: teacher-{_r20["steps"]} agr {_m20["agreement"]:.3f} ({_r20["nfe"]} NFE, {_r20["time_s"]} c) | '
      f'teacher-{_r10["steps"]} agr {_m10["agreement"]:.3f} ({_r10["nfe"]} NFE, {_r10["time_s"]} c) | '
      f'student-{_rs["steps"]} agr {_ms["agreement"]:.3f} ({_rs["nfe"]} NFE, {_rs["time_s"]} c) | '
      f'L2 к teacher-20: student {_l2_stu:.3f} против teacher-10 {_l2_t10:.3f} | '
      f'PNG: lab18_teacher20/teacher10/student10.png, lab18_worst_generated.png | отчёт: lab18_student_vs_teacher.json')

ЗАДАНИЕ 10 OK: teacher-20 agr 1.000 (40 NFE, 0.7 c) | teacher-10 agr 1.000 (20 NFE, 0.4 c) | student-10 agr 1.000 (20 NFE, 0.4 c) | L2 к teacher-20: student 10.425 против teacher-10 10.467 | PNG: lab18_teacher20/teacher10/student10.png, lab18_worst_generated.png | отчёт: lab18_student_vs_teacher.json


### Творческое мини-исследование

*Вес: 1 балл, бонус сверх десяти обязательных заданий.*

**Постановка.** Контролируемая абляция позволяет отделить влияние одного фактора от шума выборки. Гипотеза формулируется до просмотра отложенных результатов.

Выполните следующие действия:

1. Сформулируйте собственную предметную гипотезу о поведении модели и измените один фактор, сохраняя те же реальные входы, seed и вычислительный бюджет.
2. Сравните с основным экспериментом, не используя test для выбора настройки; объясните наблюдаемые расхождения.

**Результат.** Отдельные код, измеренная таблица и интерпретация в отчёте.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена на реальных входах, сохранены фактические измерения и показана контролируемая разница. Бонус не является обязательным для Run All.

In [14]:
# БОНУС (1 балл сверх десяти обязательных) - творческое мини-исследование.
# Гипотеза (сформулирована ДО просмотра результатов): при равном малом бюджете (5 шагов, w=3,
# общий x_T и общий seed) DPM-Solver++(2M) сохранит class agreement БЛИЖЕ к длинному DDIM-50,
# чем DDIM-5, потому что точнее решает то же самое ОДУ обратного процесса; разнообразие при
# этом изменится слабо, так как начальный шум и детерминизм у обоих samplers одинаковы.
# Контроль: изменён ровно ОДИН фактор - тип solver (DDIM против DPM-Solver++(2M)); модель
# (EMA teacher), x_T, метки, seed, число шагов и guidance фиксированы. Test не используется.
BONUS_STEPS = 2 if FAST_DEV_RUN else 5
REF_STEPS = 4 if FAST_DEV_RUN else 50
_ref_imgs, _ref_nfe = ddim_sample(TEACHER, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT, ddim_grid(REF_STEPS))
_ddim_imgs, _ddim_nfe = ddim_sample(TEACHER, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT, ddim_grid(BONUS_STEPS))
_dspp_imgs, _dspp_nfe = dpm_solver_pp_sample(TEACHER, GEN_LABELS.to(DEVICE), GUIDANCE_W, X_T_INIT,
                                             ddim_grid(BONUS_STEPS), order=2)
_ref_imgs, _ddim_imgs, _dspp_imgs = _ref_imgs.cpu(), _ddim_imgs.cpu(), _dspp_imgs.cpu()

def _m(imgs):
    _agr, _conf = agreement_conf(imgs)
    _div = intra_class_diversity(imgs, GEN_LABELS)
    return {'agreement': round(_agr, 4), 'confidence': round(_conf, 4),
            'diversity_intra_l2': None if _div is None else round(_div, 3)}

_m_ref, _m_ddim, _m_dspp = _m(_ref_imgs), _m(_ddim_imgs), _m(_dspp_imgs)
_l2 = lambda a, b: round(float(torch.cdist(a.reshape(a.size(0), -1), b.reshape(b.size(0), -1)).mean()), 3)
_drop_d = round(_m_ref['agreement'] - _m_ddim['agreement'], 4)
_drop_s = round(_m_ref['agreement'] - _m_dspp['agreement'], 4)
_verdict = 'ПОДТВЕРЖДЕНА' if _drop_s < _drop_d else 'ОПРОВЕРГНУТА'
save_json('lab18_bonus_solver.json', {
    'hypothesis': 'DPM-Solver++(2M) при 5 шагах сохранит class agreement ближе к DDIM-50, чем DDIM-5 '
                  '(точнее решение того же ОДУ); разнообразие изменится слабо',
    'fixed': {'w': GUIDANCE_W, 'model': 'EMA teacher (лаба 17)', 'x_T_seed': X_T_SEED,
              'sampling_seed': SAMPLING_SEED, 'steps': BONUS_STEPS},
    'factor': 'тип solver: DDIM против DPM-Solver++(2M) при равном бюджете',
    'reference': {'sampler': f'ddim-{REF_STEPS}', **_m_ref, 'nfe': _ref_nfe},
    'ddim': {'steps': BONUS_STEPS, **_m_ddim, 'nfe': _ddim_nfe},
    'dspp2m': {'steps': BONUS_STEPS, **_m_dspp, 'nfe': _dspp_nfe},
    'agreement_drop_vs_reference': {'ddim': _drop_d, 'dspp2m': _drop_s},
    'l2_to_reference_mean': {'ddim': _l2(_ddim_imgs, _ref_imgs), 'dspp2m': _l2(_dspp_imgs, _ref_imgs)},
    'verdict': _verdict})
print(f'БОНУС OK: DDIM-{REF_STEPS} agreement {_m_ref["agreement"]:.3f} (NFE {_ref_nfe}) | при {BONUS_STEPS} шагах: '
      f'DDIM {_m_ddim["agreement"]:.3f} (падение {_drop_d:.3f}) против DPM-Solver++(2M) {_m_dspp["agreement"]:.3f} '
      f'(падение {_drop_s:.3f}) | NFE: ddim {_ddim_nfe}, dspp {_dspp_nfe} | '
      f'L2 к reference: ddim {_l2(_ddim_imgs, _ref_imgs)}, dspp {_l2(_dspp_imgs, _ref_imgs)} | '
      f'гипотеза: {_verdict} | отчёт: lab18_bonus_solver.json')

БОНУС OK: DDIM-50 agreement 1.000 (NFE 100) | при 5 шагах: DDIM 1.000 (падение 0.000) против DPM-Solver++(2M) 0.963 (падение 0.037) | NFE: ddim 10, dspp 10 | L2 к reference: ddim 10.418, dspp 10.409 | гипотеза: ОПРОВЕРГНУТА | отчёт: lab18_bonus_solver.json


## Анализ результатов

Сопоставьте конфигурации при одинаковых данных, seeds и бюджете. Приведите таблицу фактических метрик и времени, не менее пяти неудачных примеров с объяснением причин, ограничения и проверяемые выводы. Не подменяйте измерения утверждением об успешном запуске.

## Требования к сдаче

Перед сдачей перезапустите ядро и выполните все ячейки по порядку в полном режиме. Приложите конфигурацию, версии библиотек, источники/лицензии данных и модели, checkpoints, таблицу измерений, изображения и краткий анализ ошибок. Для длительного обучения перенесите каталог `artifacts/` на Google Drive и возобновляйте обучение с последнего checkpoint; отсутствие GPU или данных нельзя заменять синтетическими метриками.

## Итоги (cv18-summary)

**Протокол.** Устройство: cuda — NVIDIA GeForce GTX 1050 Ti (cc 6.1). Версии: torch 2.14.1+cu126, torchvision 0.29.1+cu126, numpy 2.4.6, matplotlib 3.11.2. Режим: полный (FAST_DEV_RUN=False), seed 42, cudnn deterministic — сравнение samplers воспроизводится цифра в цифру; smoke и full физически разделены (`artifacts/full`).

**Teacher и протокол сравнения (задания 1–2).** Teacher — checkpoint лабы 17 `lab17_ckpt_denoiser.pt` (sha256 2eb05e6c7f16.., step 10000); T=500 взят из checkpoint и зафиксирован, EMA-веса, 415 041 параметр, заморожен на всей лабе. Проверяющий классификатор DigitCNN 20 490 параметров: test accuracy по отчёту лабы 17 0.9770, независимо перепроверено на тех же 2000 test-изображениях — 0.9770 (дрейф 0.0000), т.е. checkpoint подлинный. Протокол: 80 изображений (по 8 на класс 0..9), общий x_T (seed 92) и общий пошаговый шум (seed 93) у всех samplers, guidance w=3.0.

**Задания 3–5 (корректность samplers и детерминизм).** Сетка DDIM строго убывающая с концами T-1=499 и 0; формула перехода сверена с аналитикой на псевдо-сети с известным выходом; граница s=-1 (alpha_bar=1) даёт чистый x0_hat. DPM-Solver++(2M) построен на том же denoiser и том же beta schedule: порядок 1M пошагово совпал с DDIM на общей сетке (проверено), 2M на 4 шагах от DDIM отклоняется — max |diff| 1.0000 при NFE 8. Детерминизм: два запуска DDIM (сетка 20 шагов, NFE 40) дали побитово одинаковые изображения, состояние RNG (CPU и CUDA) не изменилось; время 0.8 c / 0.7 c.

**Задание 6 (бюджеты sampling).** Девять прогонов на одном batch с общим шумом и классами; NFE учитывает conditional и unconditional вызовы отдельно (при w=3 — 2 вызова/шаг):

| Sampler | шагов | agreement | уверенность | L2 внутри класса | NFE | время |
|---|---|---|---|---|---|---|
| DDPM | 500 | 0.988 | 0.991 | 8.002 | 1000 | 18.0 c |
| DDIM | 50 | 1.000 | 1.000 | 8.123 | 100 | 1.8 c |
| DS++(2M) | 50 | 1.000 | 1.000 | 8.307 | 100 | 1.8 c |
| DDIM | 20 | 1.000 | 1.000 | 7.899 | 40 | 0.7 c |
| DS++(2M) | 20 | 1.000 | 0.999 | 8.417 | 40 | 0.7 c |
| DDIM | 10 | 1.000 | 1.000 | 7.537 | 20 | 0.4 c |
| DS++(2M) | 10 | 1.000 | 0.999 | 8.511 | 20 | 0.4 c |
| DDIM | 5 | 1.000 | 1.000 | 6.735 | 10 | 0.2 c |
| DS++(2M) | 5 | 0.962 | 0.958 | 8.663 | 10 | 0.2 c |

Ключевое: DDIM-5 сохраняет agreement 1.000 при NFE 10 против 1000 у DDPM (×100) и 0.2 c против 18.0 c (×90); цена — сжатие внутриклассового разнообразия (6.735 против 8.002). DPM-Solver++(2M) не превзошёл DDIM ни в одном бюджете, а на 5 шагах деградировал (0.962).

**Задание 7 (Pareto).** По измеренным времени и agreement недоминируемая точка одна — **ddim-s5** (0.2 c, agreement 1.000): остальные прогоны либо дороже при том же качестве, либо хуже качеством. Фронт вырождается в единственную точку — сильный аргумент в пользу коротких DDIM-сеток на этой задаче.

**Задания 8–10 (progressive distillation).** Student — отдельный TinyDenoiser 415 041 параметр, инициализирован весами teacher (max |diff| 0.0), teacher заморожен; train MNIST — тот же фиксированный префикс 20 000, что в лабе 17. Цель: один переход student аппроксимирует два DDIM-перехода teacher (w=3) на сетке 20 точек (10 пар). Обучение: 1500 шагов, батч 128, lr 2e-4, EMA — MSE 0.0000 → 0.0001 (avg50 в конце ≈ 0.0005), время 492.1 c; чекпоинт `artifacts/full/lab18_ckpt_student.pt` (старый общий путь из корня artifacts корректно проигнорирован — smoke/full не смешиваются). Сравнение при равных NFE: teacher-20 — agreement 1.000 (40 NFE, 0.7 c), teacher-10 — 1.000 (20 NFE, 0.4 c), student-10 — 1.000 (20 NFE, 0.4 c); близость к 20-шаговому эталону: student-10 L2 10.425 против teacher-10 L2 10.467 — student при том же бюджете не уступает teacher и чуть ближе к эталону. Неудачные примеры для анализа: `lab18_worst_generated.png` — пять слабейших генераций student-10 по уверенности классификатора с целевым и предсказанным классом.

**Бонус (DS++(2M) на малом бюджете).** Гипотеза сформулирована до измерений: «DPM-Solver++(2M) при 5 шагах сохранит class agreement ближе к длинному DDIM-50, чем DDIM-5, потому что точнее решает то же ОДУ; разнообразие изменится слабо». Изменён ровно один фактор — тип solver; модель (EMA teacher), x_T, метки, seed, шаги и guidance фиксированы. Результат против DDIM-50 (agreement 1.000, NFE 100): DDIM-5 — 1.000 (падение 0.000), DS++(2M)-5 — 0.963 (падение 0.037); L2 к reference: 10.418 против 10.409. Гипотеза **опровергнута**: при равном NFE точность agreement у 2M хуже, а разнообразие и близость к reference практически не отличаются. Вероятное объяснение (предположение): экстраполяция 2M по двум последним x0-предсказаниям чувствительна к ошибкам eps на грубой сетке — согласуется с расхождением 2M от DDIM уже на 4 шагах (max |diff| 1.0000).

**Время этапов.** Дистилляция 492.1 c — единственный дорогой этап; девять бюджетных прогонов суммарно ≈ 24.2 c (сумма измеренных времён); teacher-проверки и детерминизм — доли секунды. Артефакты в `artifacts/full/`: отчёты `lab18_teacher_check / protocol / ddim_grid / dpm_solver / determinism / budgets / pareto / distillation / student_vs_teacher / bonus_solver` (.json), чекпоинт `lab18_ckpt_student.pt`, PNG: `lab18_budget_*.png` (9 сеток), `lab18_pareto.png`, `lab18_teacher20.png`, `lab18_teacher10.png`, `lab18_student10.png`, `lab18_worst_generated.png`.

**Вывод.** На компактной DDPM (T=500, 415k параметров) сокращение траектории почти бесплатно: DDIM на 5 шагах (NFE 10, 0.2 c) держит agreement 1.000 — ×100 по NFE и ×90 по времени против полной цепи DDPM, а Pareto-фронт вырождается в ddim-s5. DPM-Solver++(2M) преимуществ на этой задаче не показал. Прогрессивная дистилляция «2 перехода teacher → 1 переход student» дала student'а, который при равном NFE не уступает teacher'у и чуть ближе к 20-шаговому эталону (L2 10.425 против 10.467). Примечательно на стыке лаб: в лабе 17 детерминированный DDIM-проход по всем шагам дал agreement 0.000, здесь же DDIM стабилен на всех сетках — по-видимому, решает корректная финальная проекция на x0_hat (граница s=-1), а не детерминизм сам по себе. Каждое число — измерение с фиксированными seed'ами, воспроизводимое перезапуском ядра, а не утверждение об успешном запуске.